# Lab 12: Segmentation and Zones, a Free-or-Taken Checker

**Duration:** 3 hours. **Level:** beginner. **Work mode:** pairs or individual.

| Student name | Station no. | Date |
|---|---|---|
| | | |

## 1. Overview
A detector draws a **box** around an object. But a box is a poor outline for a person who sits on a chair or a car that is parked at an angle: most of the box is empty background. An **instance segmentation** model finds the **exact shape** (the **mask**) of every object. With the shape you can ask a better question: how much of a **zone** is covered by an object? In this lab you mark a zone in the picture, for example a parking bay or a seat, and the program tells you if it is **FREE** or **TAKEN**. Every change is saved in a log with the time.

**By the end of this lab you can:**
1. Explain the difference between detection (box) and segmentation (mask), and when a mask is better.
2. Read the mask of an object as a list of points (a polygon) and calculate its area.
3. Define a zone in the picture and calculate how much of it is covered.
4. Use a hold time to stop the status from flickering.
5. Test the checker and explain why it sometimes gives a wrong status.

**How this notebook works.** Run a cell with **Shift + Enter**, from the top. Type your answers in the cells that say *Your answers*. The live video appears under the cell that starts it. Press **Ctrl + S** often.

> **Privacy.** The camera sees people. Use it only on yourself and on classmates who agree. The program does not save pictures.

## 2. Background

**Detection and segmentation.** The model `yolo26n-seg` finds the objects like the detector of Lab 10, but for each object it also gives a **mask**: the pixels that belong to the object. The mask comes as a **polygon**, a list of (x, y) corner points that go around the object.

| | Detection | Segmentation |
|---|---|---|
| Result for one object | Class, confidence and a **box** (4 numbers) | Class, confidence, a box and a **polygon** (many points) |
| Shape | Always a rectangle | The real outline |
| Speed | Faster | Slower, because the model does more |
| Good for | Counting objects | Measuring how much space an object takes |

**Zones.** A zone is a shape that you draw in the picture, here a polygon of corner points. You write the corners as fractions of the picture (0 is the left or top edge, 1 is the right or bottom edge), so the zone fits any picture size.

**Coverage.** Coverage tells how much of the zone is covered by the objects you care about:

`coverage = (pixels that are in the zone AND in an object mask) / (pixels in the zone)`

A coverage of 0 means the zone is empty. A coverage of 1 means it is completely covered. The status is **TAKEN** when the coverage is at least `TAKEN_AT` (for example 20 %).

| Box or mask? | A person sits on a chair in the corner of the zone |
|---|---|
| Box | The box also covers the empty area next to the person, so the zone looks taken too early |
| Mask | Only the pixels of the person count, so the coverage is honest |

**The hold time.** For one picture the model can miss an object, or see a ghost. If the status changed on every picture, it would flicker. The program changes the status only when the **new** status has lasted for `HOLD` seconds in a row. It is the same idea as the cooldown of the QR attendance in Lab 7 and the margin you will meet in Lab 14: a small gap or delay makes a stable system.

| Setting | If you raise it | If you lower it |
|---|---|---|
| `TAKEN_AT` | A zone must be more covered to count as taken | A small corner of an object already makes it taken |
| `HOLD` | The status is calmer but reacts later | The status reacts fast but can flicker |

**Change one thing at a time.** If you break the notebook, close it and run `import labview; labview.reset(12)` in a cell of a new notebook (File, New, Notebook) for a fresh copy.

---
# Part A: Prepare and Plan (20 min)

### Step 1: Get the Model File
The model file `yolo26n-seg.pt` is small (about 8 MB). The cell uses the copy in the `models` folder, or the model is downloaded the first time you load it in Part B (this needs the internet).

In [ ]:
import glob
import os
import shutil

print("This notebook is saved in:", os.getcwd())

for source in ["../models/yolo26n-seg.pt", "../lab13/yolo26n-seg.pt"]:
    if os.path.exists(source) and not os.path.exists("yolo26n-seg.pt"):
        shutil.copy(source, "yolo26n-seg.pt")
        print("copied", source)

if os.path.exists("yolo26n-seg.pt"):
    print("The segmentation model is here.")
else:
    print("The model will be downloaded when you load it in Part B (needs the internet).")

### Step 2: Plan Your Checker
Choose what the zone is and what can take it. The class names are the 80 classes of Lab 10.

| Idea | Zone | Objects that take it (`TARGETS`) |
|---|---|---|
| Parking bay | A bay on the floor or a drawn rectangle on the table | car (use toy cars or a picture of a car on a phone) |
| Seat checker | One chair | person |
| Desk tidy | The space in front of you | cell phone, bottle, cup |
| Pet bed | A bed or a mat | cat, dog |

**Your answers** (double-click this cell, type after each colon, then press Shift + Enter):

- Your idea:
- What is the zone and where is the camera?:
- Which class names take the zone (TARGETS)?:
- What should a real user do when the status is TAKEN or FREE?:


---
# Part B: See the Masks (35 min)

### Step 1: The Settings
Words in capital letters are settings you can change. Run the cell. The `ZONE` is a list of corner points, each as (x, y) fractions of the picture. The first zone is the right half of the picture, a rectangle with four corners.

In [ ]:
# ---------------- Settings you can change ----------------
TARGETS = ["person"]     # the classes that take the zone (any of the 80 class names)
ZONE = [(0.55, 0.25), (0.95, 0.25), (0.95, 0.95), (0.55, 0.95)]   # corners: (x, y) as fractions, going around the zone
TAKEN_AT = 20            # the zone is TAKEN when this percent or more of it is covered
HOLD = 2.0               # seconds a new status must last before it is shown
CONFIDENCE = 0.40        # confidence limit
MODEL = "yolo26n-seg.pt"
# ----------------------------------------------------------
print("Targets:", TARGETS, "| taken at", TAKEN_AT, "% | hold", HOLD, "s")

### Step 2: Load the Model

In [ ]:
import csv
import time
from datetime import datetime

import cv2
import numpy as np
import torch
from ultralytics import YOLO

import labview

device = 0 if torch.cuda.is_available() else "cpu"
model = YOLO(MODEL)
print("Using model:", MODEL, "on", "the GPU" if device == 0 else "the CPU")

### Step 3: Box and Mask in Numbers
Run the model on a picture and compare, for every object, the area of the **box** with the area of the **mask**. `result.masks.xy` is a list with one polygon for each object, in pixels of the picture. The area of a polygon is calculated with `cv2.contourArea`.

In [ ]:
picture = next((p for p in ("../models/bus.jpg", "../lab15/bus.jpg") if os.path.exists(p)), None)
if picture is None:
    print("No sample picture found. Skip this step.")
else:
    result = model(picture, conf=CONFIDENCE, device=device, verbose=False)[0]
    print("Objects found:", len(result.boxes), "| masks:", len(result.masks.xy))
    print()
    print(f"{'object':<12}{'box area':>10}{'mask area':>11}{'mask / box':>12}{'polygon points':>16}")
    for box, polygon, cls in zip(result.boxes.xyxy.tolist(), result.masks.xy, result.boxes.cls):
        box_area = (box[2] - box[0]) * (box[3] - box[1])
        mask_area = cv2.contourArea(polygon.astype(np.float32))
        print(f"{result.names[int(cls)]:<12}{box_area:>10.0f}{mask_area:>11.0f}{mask_area / box_area:>12.2f}{len(polygon):>16}")

**Your answers** (double-click this cell, type after each colon, then press Shift + Enter):

- How many objects did the model find?:
- For a person, what is the ratio of mask area to box area?:
- For the bus, is the ratio higher or lower than for a person? Why?:
- How many points does the polygon of one object have?:


### Step 4: The Masks Live
`result.plot(boxes=False)` colours the mask of every object. The program also writes the mask area as a percentage of the picture. Run the cell, show your hand, your face and some objects, and watch the masks follow their edges. Press **Stop** when you are done.

In [ ]:
def process_masks(frame, state):
    result = model(frame, conf=state["Confidence"], device=device, verbose=False)[0]
    shown = result.plot(boxes=False)
    total = frame.shape[0] * frame.shape[1]
    y = 30
    if result.masks is not None:
        for polygon, cls in zip(result.masks.xy, result.boxes.cls):
            percent = 100 * cv2.contourArea(polygon.astype(np.float32)) / total
            cv2.putText(shown, f"{result.names[int(cls)]}: {percent:.1f} % of the picture", (10, y),
                        cv2.FONT_HERSHEY_SIMPLEX, 0.6, (255, 255, 255), 2)
            y += 26
    cv2.putText(shown, f"{state['fps']:.0f} FPS", (shown.shape[1] - 110, 30), cv2.FONT_HERSHEY_SIMPLEX, 0.6, (255, 255, 255), 1)
    return shown


live = labview.live(process_masks, controls={"Confidence": (0.05, 0.95, 0.05, CONFIDENCE)})

**Your answers** (double-click this cell, type after each colon, then press Shift + Enter):

- FPS shown on the screen (it is lower than in Lab 10 because the model does more work):
- What happens to the percent when you move your hand closer to the camera?:
- Where is the mask not accurate (edges, thin parts, hair)?:


---
# Part C: Build the Free-or-Taken Checker (55 min)

### Step 1: The Program
Read this cell before you run it. The new parts are:

| Function | What it does |
|---|---|
| `zone_polygon(shape)` | Turns the `ZONE` fractions into pixel corners for a picture of this size |
| `coverage_of(result, shape)` | Draws the zone and the masks of the `TARGETS` as two black and white pictures and returns the share of the zone that is covered (0 to 1) |
| `update_state(taken_now, now)` | The hold rule: changes the status only after the new status lasted `HOLD` seconds. It returns the old status and how long it lasted when the status changes, otherwise `None`. |
| `log_change(before, spent, percent)` | Adds a line to `zone_log.csv` and rewrites `zone_log.xlsx` |
| `process(frame, state)` | Runs the model, measures the coverage, updates the status and draws. The live view calls it for every picture. |

Find the five steps again: capture (live view), detect (the model finds the masks), decide (`coverage_of`, `update_state`), act (the drawing) and record (`log_change`).

In [ ]:
run = {"state": "FREE", "since": time.time(), "candidate": None, "cand_since": 0.0, "percent": 0.0, "note": "", "warned": False}


def zone_polygon(shape):
    """The ZONE in pixels for a picture of this shape (height, width, ...)."""
    h, w = shape[:2]
    return np.array([[int(x * w), int(y * h)] for x, y in ZONE], dtype=np.int32)


def coverage_of(result, shape):
    """Share of the zone (0 to 1) that is covered by the masks of the TARGETS."""
    zone = np.zeros(shape[:2], np.uint8)
    cv2.fillPoly(zone, [zone_polygon(shape)], 1)               # 1 inside the zone
    objects = np.zeros(shape[:2], np.uint8)
    if result.masks is not None:
        for polygon, cls in zip(result.masks.xy, result.boxes.cls):
            if result.names[int(cls)] in TARGETS and len(polygon) >= 3:
                cv2.fillPoly(objects, [polygon.astype(np.int32)], 1)     # 1 inside an object
    covered = int(np.count_nonzero(zone & objects))                      # in the zone AND in an object
    return covered / max(1, int(np.count_nonzero(zone)))


def update_state(taken_now, now):
    """The hold rule. Returns (old status, seconds it lasted) when the status changes, otherwise None."""
    wanted = "TAKEN" if taken_now else "FREE"
    if wanted == run["state"]:
        run["candidate"] = None                                 # nothing to change
        return None
    if run["candidate"] != wanted:
        run["candidate"], run["cand_since"] = wanted, now        # start to wait
        return None
    if now - run["cand_since"] >= HOLD:                         # the new status lasted long enough
        before, spent = run["state"], now - run["since"]
        run.update(state=wanted, since=now, candidate=None)
        return before, spent
    return None


def write_excel():
    """Rewrite zone_log.xlsx from zone_log.csv, so it opens in Excel with a header and number columns."""
    try:
        from openpyxl import Workbook
        from openpyxl.styles import Font
    except ImportError:
        if not run["warned"]:
            run["warned"] = True
            print("zone_log.xlsx needs openpyxl. zone_log.csv works in Excel too. Ask your instructor.")
        return
    book = Workbook()
    sheet = book.active
    sheet.title = "Zone"
    with open("zone_log.csv", newline="") as f:
        rows = list(csv.reader(f))
    for i, row in enumerate(rows):
        if i > 0:
            row[2], row[3] = float(row[2]), float(row[3])
        sheet.append(row)
    for cell in sheet[1]:
        cell.font = Font(bold=True)
    for column, width in zip("ABCD", (20, 10, 16, 26)):
        sheet.column_dimensions[column].width = width
    book.save("zone_log.xlsx")


def log_change(before, spent, percent):
    """Add one line to the log files (zone_log.csv and zone_log.xlsx)."""
    first = not os.path.exists("zone_log.csv")
    with open("zone_log.csv", "a", newline="") as f:
        writer = csv.writer(f)
        if first:
            writer.writerow(["time", "new_status", "covered_percent", "seconds_in_previous_status"])
        writer.writerow([datetime.now().strftime("%Y-%m-%d %H:%M:%S"), run["state"], round(percent, 1), round(spent, 1)])
    write_excel()


def process(frame, state):
    result = model(frame, conf=state["Confidence"], device=device, verbose=False)[0]
    shown = result.plot(boxes=False)
    fraction = coverage_of(result, frame.shape)
    percent = 100 * fraction
    now = time.time()
    change = update_state(percent >= state["Taken at (%)"], now)
    if change:
        log_change(change[0], change[1], percent)
        run["note"] = f"Changed to {run['state']} at {datetime.now().strftime('%H:%M:%S')}"

    colour = (0, 0, 255) if run["state"] == "TAKEN" else (0, 200, 0)
    cv2.polylines(shown, [zone_polygon(frame.shape)], True, colour, 3)
    cv2.putText(shown, run["state"], (10, 40), cv2.FONT_HERSHEY_SIMPLEX, 1.4, colour, 4)
    cv2.putText(shown, f"covered: {percent:.0f} %   for {now - run['since']:.0f} s", (10, 75),
                cv2.FONT_HERSHEY_SIMPLEX, 0.7, (255, 255, 255), 2)
    if run["candidate"]:
        left = max(0.0, HOLD - (now - run["cand_since"]))
        cv2.putText(shown, f"changing to {run['candidate']} in {left:.1f} s", (10, 105),
                    cv2.FONT_HERSHEY_SIMPLEX, 0.6, (0, 255, 255), 2)
    cv2.putText(shown, f"{state['fps']:.0f} FPS", (shown.shape[1] - 110, 30), cv2.FONT_HERSHEY_SIMPLEX, 0.6, (255, 255, 255), 1)
    if run["note"]:
        cv2.putText(shown, run["note"], (10, shown.shape[0] - 12), cv2.FONT_HERSHEY_SIMPLEX, 0.5, (255, 255, 255), 1)
    return shown


print("The program is ready.")

### Step 2: Check the Coverage Maths
Before the live test, check `coverage_of` with a fake result. The fake object covers the **left half** of the zone, so the coverage must be about 0.5.

In [ ]:
class Fake:
    """A fake model result with one 'person' whose mask is a rectangle (the format of result.masks.xy)."""
    names = {0: "person"}

    def __init__(self, corners):
        self.masks = type("M", (), {"xy": [np.array(corners, dtype=np.float32)]})()
        self.boxes = type("B", (), {"cls": [0]})()


shape = (480, 640, 3)
zone = zone_polygon(shape)
left, right = zone[:, 0].min(), zone[:, 0].max()
top, bottom = zone[:, 1].min(), zone[:, 1].max()
middle = (left + right) // 2
half = Fake([(left, top), (middle, top), (middle, bottom), (left, bottom)])
print("Left half covered:", round(coverage_of(half, shape), 2), "(expected about 0.5)")
print("Nothing there:    ", round(coverage_of(Fake([(0, 0), (5, 0), (5, 5)]), shape), 2), "(expected 0.0)")

**Your answers** (double-click this cell, type after each colon, then press Shift + Enter):

- Did both lines give the expected numbers?:
- Why do we divide by the number of pixels of the zone?:


### Step 3: Run the Checker
Run the cell. A coloured outline shows the zone: green is FREE and red is TAKEN. Step into the zone, stay for a few seconds and leave again. Watch the **coverage**, the yellow text `changing to ... in ... s` (the hold time counts down) and the status. Use the sliders to change the confidence limit and the percent at which the zone counts as taken. Press **Stop** when you are done.

In [ ]:
run.update(state="FREE", since=time.time(), candidate=None, cand_since=0.0, percent=0.0, note="")
live = labview.live(process, controls={"Confidence": (0.05, 0.95, 0.05, CONFIDENCE),
                                       "Taken at (%)": (5, 90, 5, TAKEN_AT)})

### Step 4: See the Results
Run this cell after you pressed Stop. It shows the log and the total time the zone was taken. To open the Excel file, use the file browser on the left of JupyterLab, right-click `zone_log.xlsx` and choose **Download**, then open it on your laptop.

In [ ]:
def show_results():
    for name in ("zone_log.csv", "zone_log.xlsx"):
        print(f"{name:<15}", "found" if os.path.exists(name) else "not created yet (it appears at the first change of status)")
    if os.path.exists("zone_log.csv"):
        with open("zone_log.csv", newline="") as f:
            rows = list(csv.DictReader(f))
        taken = sum(float(r["seconds_in_previous_status"]) for r in rows if r["new_status"] == "FREE")
        print()
        print("Changes of status:", len(rows))
        print(f"Time the zone was taken (completed periods): {taken:.0f} seconds")
        print()
        with open("zone_log.csv") as f:
            print(f.read())


show_results()

**Your answers** (double-click this cell, type after each colon, then press Shift + Enter):

- Coverage shown when you stood fully in the zone:
- Coverage shown when only your arm was in the zone:
- How long did it take from stepping in to TAKEN? Does this match HOLD?:
- First line of the log after the header:


---
# Part D: Make It Your Own (30 min)
Change `TARGETS` and `ZONE` to match your plan from Part A. To draw your own zone, estimate the corners in the picture: the top left is (0, 0), the bottom right is (1, 1). A zone does not have to be a rectangle, for example a triangle needs three corners. Then add **at least one improvement** from the list, or one of your own. Change the code in the cells above, run that cell again, then run Step 3 of Part C again.

- Use two zones (for example two parking bays) and show the status of each.
- Show how long the zone has been FREE or TAKEN in minutes and seconds.
- Count how many times the zone became TAKEN today and show it on the screen.
- Make the outline yellow when the coverage is between 5 % and `TAKEN_AT` (a person is entering).
- Add the class name of the object that covers most of the zone to the log.
- Change `HOLD` to 0.5 and to 6, and describe the difference.

> **Tip.** Change one thing at a time and run it after each change. To keep a copy of a cell before a big change, copy the cell (select it, press **C**, then **V**) and change the copy.

**Your answers** (double-click this cell, type after each colon, then press Shift + Enter):

- Your TARGETS and ZONE (the corner list):
- Improvement you chose:
- Which lines did you change?:
- Did it work the first time? What went wrong?:


---
# Part E: Test and Present (40 min)

### Step 1: Test the Checker
Delete the old log, run the checker again and try each situation. Write what the program showed.

In [ ]:
# Delete the old log to start a clean test
for name in ("zone_log.csv", "zone_log.xlsx"):
    if os.path.exists(name):
        os.remove(name)
print("Clean. Files left:", sorted(glob.glob("zone_log*")))

Run Step 3 of Part C for each test (it starts with the status FREE).

| Test | Situation | Expected status | What happened (status and coverage) |
|---|---|---|---|
| 1 | The zone is empty for 10 seconds | FREE | |
| 2 | A target stands fully in the zone for 10 seconds | TAKEN after about `HOLD` seconds | |
| 3 | A target stands half in the zone | Depends on `TAKEN_AT` | |
| 4 | A target walks through the zone in 1 second (less than `HOLD`) | FREE (no change) | |
| 5 | A non-target object (not in `TARGETS`) is in the zone | FREE | |
| 6 | The target leaves the zone | FREE after about `HOLD` seconds | |
| 7 | Dim light | Compare with test 2 | |

**Score** = the number of tests where the status was as expected, out of 7.

**Your answers** (double-click this cell, type after each colon, then press Shift + Enter):

- Test 1 result:
- Test 2 result and time to TAKEN:
- Test 3 result and coverage:
- Test 4 result: did the hold time prevent a change?:
- Test 5 result:
- Test 6 result:
- Test 7 result:
- Score out of 7:
- TAKEN_AT and HOLD values that worked best:


### Step 2: Report and Present
Fill in the report. Then give a **3-minute demonstration** to your instructor: show the masks, the zone, the change from FREE to TAKEN and back, the log in Excel, and explain the coverage and the hold time.

**Your answers** (double-click this cell, type after each colon, then press Shift + Enter):

- Idea, TARGETS and ZONE:
- Model file used and the FPS shown:
- Power mode used (run  !sudo nvpmodel -q  in a cell, after  labview.sudo_login()  ):
- Your improvement:
- Score of the test table:
- One problem you found and how you fixed it:
- One idea to make it better in future:


> **Where next.** A zone checker is the base of parking systems, smart desks and shelf monitors. In the next lab you use a model that finds the **body points** of a person (pose). In Lab 32 you will teach a model **new** objects that you choose yourself.

## 4. Check Task
Show your instructor. The instructor initials each line when it is done correctly.

| No. | Task to demonstrate | Initials |
|---|---|---|
| 1 | The masks show live and you explain the difference to a box | |
| 2 | The numbers of Part B Step 3 are explained (box area, mask area) | |
| 3 | The two lines of the coverage check give the expected numbers | |
| 4 | The zone outline is green when FREE and red when TAKEN | |
| 5 | The hold time is visible (`changing to ... in ... s`) and the status does not flicker | |
| 6 | The log is shown in Excel (`zone_log.xlsx`) with at least 2 lines | |
| 7 | At least one improvement works and you explain the code you changed | |
| 8 | The test table in Part E is complete with the score | |
| 9 | The report is complete and the 3-minute presentation is given | |

## 5. Assessment Questions
Answer in one or two sentences.

**Your answers** (double-click this cell, type after each colon, then press Shift + Enter):

- 1. What does a segmentation model give that a detection model does not?:
- 2. What is a polygon, and how does the program get the area of one?:
- 3. Explain the formula of coverage in your own words.:
- 4. Why is a mask better than a box when you ask 'is the seat taken?':
- 5. Why do we write the zone as fractions and not as pixels?:
- 6. What is the hold time for? What happens if `HOLD = 0`?:
- 7. The zone shows TAKEN when a person only walks past. Which setting do you change, and in which direction? What is the cost?:
- 8. Why is the segmentation model slower than the detection model?:
- 9. Which task needs a mask and not only a box? (a) Counting people in a room (b) Measuring how much of a parking bay a car covers (c) Reading a number plate (d) Finding the nose of a person:


## 6. Troubleshooting

| Symptom | Likely cause | What to try |
|---|---|---|
| The model cannot be downloaded | No internet on the Jetson | Ask your teacher. Copy `yolo26n-seg.pt` to the `models` folder. |
| `Cannot open the camera` | Another program uses the camera, or the cable is loose | Press Stop in other live cells, shut down other kernels, ask your teacher to check your camera on the teacher page |
| No colours (masks) appear | Nothing is found, or the confidence limit is too high | Show a person or a bottle. Lower the confidence slider. |
| `AttributeError ... masks` | A detection model is loaded instead of the segmentation model | Check that `MODEL = "yolo26n-seg.pt"` (with `-seg`) and run Step 2 again |
| The coverage is always 0 | The class is not in `TARGETS`, or the zone is outside the picture | Check the spelling against the Lab 10 class names. Check the `ZONE` numbers (between 0 and 1). |
| The zone is not drawn where I want it | The corners are in the wrong order or wrong place | Remember (0, 0) is the top left. List the corners in a circle around the zone. |
| The status flickers | `HOLD` is too short, or the object is partly hidden | Raise `HOLD`. Raise or lower `TAKEN_AT`. |
| The status reacts too late | `HOLD` is too long | Lower `HOLD` |
| `CUDA out of memory` | Other notebooks and programs use the shared 8 GB | Press Stop, shut down other kernels (Kernel, Shut Down All Kernels) and run again |
| The notebook looks broken | You changed or deleted too much | Close it and run `import labview; labview.reset(12)` in a cell of a new notebook (File, New, Notebook). You get a fresh copy. |

## 7. Further Reading
- Ultralytics, Segment task: https://docs.ultralytics.com/tasks/segment/
- Ultralytics, Object counting in regions: https://docs.ultralytics.com/guides/region-counting/
- OpenCV, Contour features (area): https://docs.opencv.org/4.x/dd/d49/tutorial_py_contour_features.html
- openpyxl, working with Excel files: https://openpyxl.readthedocs.io/